# ROCIO-IBEB: mean precipitation
***

***Author:** Chus Casado Rodríguez*<br>
***Date:** 24-09-2026*<br>

**Introduction:**<br>

This notebook creates a map of mean daily precipitation over Spain using the ROCIO-IBEB dataset.

**To be improved**

In [1]:
from pathlib import Path
import numpy as np

import logging
logger = logging.getLogger(__name__)

from ocab.basins.stats import read_data

import warnings
warnings.filterwarnings('ignore', message='invalid value encountered in sqrt', category=RuntimeWarning)

## Configuration

In [2]:
# meteorology
path_data = Path('/home/casadoj/Data/')
meteo = 'ROCIO-IBEB'
zarr_store = f'{meteo}_1979-2022.zarr'

# output
path_out = path_data / meteo / 'GIS'
path_out.mkdir(parents=False, exist_ok=True)

## Data

In [3]:
# load meteorological data
zarr_store = path_data / meteo / zarr_store
if zarr_store.is_dir():
    data = read_data(zarr_store)
    print(f"{data.nbytes / 1e9:.2f} GB")
    # rewrite temperature units
    for var in ['mintemp', 'maxtemp']:
        data[var].attrs['units'] = 'degC'
    # rewrite precipitation units
    data['precipitation'].attrs['units'] = 'mm/d'
else:
    logger.error(f"Zarr store doesn't exist: {zarr_store}")

12.96 GB


## Compute

In [ ]:
# compute average precipitation
precip_mean = data['precipitation'].mean('time').compute()

# # select your variable and ensure spatial dimension names are y and x
# da_precip = data['precipitation'].rename({'rlat': 'y', 'rlon': 'x'})

# assign the CRS directly from the original dataset
precip_mean.rio.write_crs(data.rio.crs, inplace=True)

# reproject to standard WGS84
precip_mean_wgs84 = precip_mean.rio.reproject("EPSG:4326")

# set nodata and export
precip_mean_wgs84.rio.write_nodata(np.nan, inplace=True)
precip_mean_wgs84.rio.to_raster(path_out / "avg_precipitation_wgs84.tif", compress="deflate")